# PINN - Barra composita 1D

Este notebook treina uma Physics-Informed Neural Network para o Projeto 1.

A formulacao usa duas sub-redes: uma para o material 1 e outra para o material 2. A interface e imposta na funcao de perda por continuidade de deslocamento e equilibrio de forca axial.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import torch
except ImportError:
    raise ImportError('PyTorch nao esta instalado. No Colab, rode: !pip install torch')

if not os.path.exists('p1_pinn.py'):
    try:
        from google.colab import files
        print('Envie o arquivo p1_pinn.py da pasta do projeto.')
        files.upload()
    except Exception:
        print('Coloque p1_pinn.py na mesma pasta deste notebook.')

from p1_pinn import BarParams, train_pinn, evaluate, write_csv, write_history, exact_displacement

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.grid'] = True

In [ ]:
def plot_pinn_result(result, params):
    x = result['x']
    interface = params.length / 2.0

    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(12, 9))
    axes[0].plot(x, result['u'], 'o-', markersize=3, label='PINN')
    axes[0].plot(x, result['u_exact'], '--', label='Analitica')
    axes[0].set_ylabel('u(x)')
    axes[0].legend()

    axes[1].plot(x, result['strain'], color='tab:orange')
    axes[1].set_ylabel('epsilon(x)')

    axes[2].plot(x, result['stress'], color='tab:green')
    axes[2].set_ylabel('sigma(x)')
    axes[2].set_xlabel('x')

    for ax in axes:
        ax.axvline(interface, color='k', linestyle=':', linewidth=1.4)

    fig.suptitle('PINN - Barra composita 1D')
    fig.tight_layout()
    plt.show()


def make_tables(result, params):
    x = result['x']
    material = np.where(x <= params.length / 2.0, 'E1', 'E2')
    table = pd.DataFrame({
        'x': x,
        'material': material,
        'u_PINN': result['u'],
        'u_analitico': result['u_exact'],
        'epsilon': result['strain'],
        'sigma': result['stress'],
        'erro_abs': np.abs(result['u'] - result['u_exact']),
    })
    return table

In [ ]:
L = widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1, description='L')
A = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='A')
E1 = widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.05, description='E1')
E2 = widgets.FloatLogSlider(value=10.0, base=10, min=-1, max=2, step=0.05, description='E2')
T = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='T')
collocation = widgets.IntSlider(value=80, min=20, max=250, step=10, description='Pontos')
adam_epochs = widgets.IntSlider(value=3000, min=500, max=10000, step=500, description='Adam')
lbfgs_steps = widgets.IntSlider(value=300, min=0, max=1000, step=100, description='LBFGS')
salvar = widgets.Checkbox(value=False, description='Salvar saidas')
treinar = widgets.Button(description='Treinar PINN', button_style='primary')
out = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([L, A, T]),
    widgets.HBox([E1, E2, collocation]),
    widgets.HBox([adam_epochs, lbfgs_steps, salvar, treinar]),
])


def on_train_clicked(_):
    with out:
        clear_output(wait=True)
        params = BarParams(length=L.value, area=A.value, e1=E1.value, e2=E2.value, traction=T.value)
        print('Treinando PINN...')
        print(f'L={params.length}, A={params.area}, E1={params.e1:.6g}, E2={params.e2:.6g}, T={params.traction}')
        model, history = train_pinn(
            params=params,
            collocation_points=collocation.value,
            adam_epochs=adam_epochs.value,
            lbfgs_steps=lbfgs_steps.value,
        )
        result = evaluate(model, params, points=201)
        print('\nResumo')
        print(f'E2/E1 = {params.e2 / params.e1:.6g}')
        print(f'u(L) PINN = {result["u"][-1]:.10g}')
        print(f'u(L) analitico = {result["u_exact"][-1]:.10g}')
        print(f'Erro relativo L2 = {result["relative_error"]:.3e}')
        print(f'Loss final = {history[-1]["total"]:.3e}')
        plot_pinn_result(result, params)
        display(make_tables(result, params).head(10))

        if salvar.value:
            os.makedirs('resultados_pinn_colab', exist_ok=True)
            write_csv(Path('resultados_pinn_colab/solucao_pinn.csv'), result)
            write_history(Path('resultados_pinn_colab/historico_treinamento.csv'), history)
            torch.save(model.state_dict(), 'resultados_pinn_colab/modelo_pinn.pt')
            print('\nArquivos salvos em resultados_pinn_colab')


treinar.on_click(on_train_clicked)
display(controls, out)

Output()

## Validacao

Para o caso do PDF, use `L=1`, `A=1`, `T=1`, `E1=1` e `E2=10`.

Valores esperados:

- `u(L) = 0.55`
- `sigma = 1`
- `epsilon1 = 1`
- `epsilon2 = 0.1`